## Tools 
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:\n",
    "1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)\n",
    "2. A function or coroutine to execute.

In [ ]:
from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model

model = init_chat_model("openai/gpt-oss-20b", model_provider="groq")
response = model.invoke("Hey, tell me about yourself?")
response

AIMessage(content='I’m ChatGPT, a large language model built by OpenAI on the GPT‑4 architecture. I was trained on a wide variety of internet text up to my last update in June\u202f2024, so I can help with questions, explanations, creative writing, coding, and many other tasks. I don’t have personal experiences or feelings—I generate responses based on patterns in the data I was trained on. My main goal is to provide useful, accurate, and friendly information. If you have anything specific in mind, just let me know!', additional_kwargs={'reasoning_content': 'We need to respond to "Hey, tell me about yourself?" as a user. We have developer message: none. The system message says we are ChatGPT. The user wants info about ourselves. According to policy, we can describe ourselves. So we can give a brief description: large language model, trained by OpenAI, GPT-4, knowledge cutoff 2024-06. We can mention capabilities: answer questions, etc. Provide friendly tone. No disallowed content. So an

In [8]:
from langchain.tools import tool


@tool
def get_weather(location: str) -> str:
    """This function get_weather returns the weather of a given location as a string"""
    return f"It's sunny in {location}"


model_with_tools = model.bind_tools([get_weather])

In [10]:
response=model_with_tools.invoke("What's the weather in gwalior like?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'The user asks: "What\'s the weather in gwalior like?" The assistant should use the get_weather function. The location is "gwalior". The assistant must call the function.', 'tool_calls': [{'id': 'fc_dfeb03dd-8538-4517-a4dd-04d66b587695', 'function': {'arguments': '{"location":"gwalior"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 137, 'total_tokens': 201, 'completion_time': 0.070130664, 'completion_tokens_details': {'reasoning_tokens': 39}, 'prompt_time': 0.009627161, 'prompt_tokens_details': None, 'queue_time': 0.049106789, 'total_time': 0.079757825}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e99e93f2ac', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a11184-ab9d-7030-993e-d898cb990684-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'gwalior'}, 'id': 'fc

### Tool Execution Loop

In [15]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."


It’s sunny in Boston.
